# Mathematical explanation of vRSA estimated using spm_reml_sc

## Introduction

In the previous notebooks, we have introduced the key idea behind vRSA at a conceptual level: patterns representation in the brain is cast as the covariance structure of the random effect of a mixed model, which is modelled in vRSA as a linear mixture of covariance components of interest. In the second notebook, we have provided a technical explanation of how the weights of the covariance matrix are estimated within the function spm_reml_sc, explaining the idea behind restricted maximal likelihood estimates, how Newton Raphson method is used to maximize it, before briefly explaining how the spm_reml_sc function instead of maximizing spm_reml_sc itself, maximizes the maximum a posteriori estimation to yield an estimate of the posterior distribution of the paramaters by incorporating priors and calculating the free energy. 

We are not fully equipped to describe what each chunck of the function spm_reml_sc does. 

## spm_reml_sc.m

### Inputs:
- `YY`: observed covariance matrix (i.e. calculated from the data), $M \times M$
- `X`: Design matrix, $M \times W$
- `Q`: covariance components, $k \times M \times M$
- `N`: Number of samples (i.e. degrees of freedom)
- `hE`: Hyperprior expectation in log space,  
- `hC`: hyperprior covariance in log space
- `V`: fixed variance component

At a conceptual level, we will try to model the covariance `YY` as a liner mixture of the $k$ covariance components in `Q`, while removing confounds associated with the fixed effects `X`, controlling for the number of degrees of freedom `N`, taking into account prior mean `hE` and covariance `hC` for the weights of each `Q` matrix. Finally, a `V` matrix can be specified, which is in case there is a covariance component we want to control for but don't want to scale with a $h_i$ parameter but still control for in our estimate of the remaining `h_i`. This is often used to model known noise or nuisance effects that are not part of the covariance components being estimated

### Line 49-79: housekeeping
The first step of the function consists in handling the inputs and initializing a few variables that will be relevant for the estimation piepline. 

- $n$ is the number of covariance components (i.e. model matrices) used to model the covariance of the data. 
- $m$ is the number of experimental condition/trials. 
- $h$ at this stage is just a matrix of 0s that will in the next steps contain the estimated $\lambda$ parameters associated with each covariance component $Q$. 
- $dFdh$ and $dFdhh$ are variables allocated for the gradient and curvature for the optimization
- $Inn$ is an identity matrix used to initialize `U` (the residual covariance matrix) 

Then from line 65 to 73, the fixed effect matrix gets orthonormalized if needs be, and the residual maker matrix is prepared. Finally, at line 76, in case a fixed covariance matrix `V` was specified but is of length 1, it gets multiplied by an identity matrix to match the dimension of the remaining covariance component matrices.

### Line 82-93: normalize components
In this step, the data, as well as the model matrices get normalized. This is a similar idea to z-scoring in a univariate model: the observed covariance of the data (i.e. YY) get divided by the magnitude of the signal in residual space. The same is done for the covariance components $Q$ to make sure that all are on comparable scales (essentially standard deviation from the mean, but in matrix terms) to ensure numerical stability and avoid certain components balloning up or escaping the regularization of the estimation algorithm.

### Line 96-112: more housekeeping

Extract the priors from the input parameters, and set to default parameters that we not provided, and set the initial values for the $\lambda$ parameters to the prior means. 

### Line 112-226: Restricted maximum likelihood and expectation maximization under variational Bayes

This is the estimation loop. 

First, from line 114-116, values are initiated: `dF`, i.e. the predicted change in free energy associated with the change in $h$, is set to infinity to ensure that the first loop proceeds. `as` is the index of the components that have not yet been deemed to be irrelevant to the model. The algorithm eliminates components that don't do anything, to make sure that they don't bias the estimation. And the `t` corresponds to the paramter that regulates the strength of the regularization. 

Finding the maximally likely estimates of $h_i$ requires calculating a couple of things. First, we have the gradient of the ReML likelihood, whose formulae is  $-\frac{N}{2}Tr(P exp(h_i) Q_i \bold{U})$. We already have `N` (which is an input to the function). The values of $h_i$ (`h(i)` in the code) are taken to be the mean of the posterior on the first iteration, and thereafter determined by the iterative optimization procedure. So we only need to calculate $U$ and $P$.

First for $P$, which is the precision-weighted projection matrix that removes the fixed effects $X$ from the data. The formulae is:

$$
P = C^{-1} - C^{-1} X (X^TC^{-1}X)^{-1}X^TC^{-1} 
$$

Where $C$ is the covariance matrix from the random effect. In the case of vRSA, $C$ is modelled as a linear combination of $Q_i$ weighted by the $h_i$ which we are trying to estimate. So in that case, $C$ is the weighted sum of these components, based on the current estimates of $h_i$, as well as the fixed covariance matrix `V` if any. This is what's calculated from line 120 to 124. Once we have $C$, we need $C^{-1}$, which is called `iC`in the code and is calculated at line 125. Then, from line 129 to 134, we calculate first $C^{-1} X$, which we call `iCX`, based of which we calculate $(X^TC^{-1}X)^{-1}$, which we call `Cq` in the code, so we have:

$$
C^{-1} - C^{-1} X (X^TC^{-1}X)^{-1}X^TC^{-1} = iC - iCX \times Cq \times iCX^T
$$

Which is calculated at line 141. Once we have $P$, we can calculate $U=I-\frac{PYY^T}{N}$, which is done at line 142. In the code, $I$ is called `Inn` and $YY^T$ is `YY` which is the first input to the function. 

With all of this calculated, we can compute the likelihood $-\frac{N}{2}Tr(P exp(h_i) Q_i \bold{U})$. Importantly, in the code, we first calculate the gradient independently of $exp(h_i)$. From line 143 to 150, we compute the derivative, and only then later in time do we bring back in $exp(h_i)$. We get one value for each covariance component, which is the gradient and the vector tells us which way is down. Once we have the gradient, we can compute the Fisher information matrix (i.e. the observed curvature), which has for formulae $\mathcal{I}_{i, j} = -\frac{N}{2}Tr(PQ_i, PQ_j)$. This is computed for each pairs of covariance components matrices from line 154 to 163 and is called `dFdhh` in the code. 

These are the two quantities derived from the observed data we use to determine which values of $h_i$ should be taken next to progress towards the maximily likely a posteriori estimates of $h_i$. However, for now, these are not expressed relative to the log of $h$, which means for now the derivative and the gradient are on the wrong scale. This is corrected for from line 166 to line 168, where the gradient and curvature get multiplied by the exponential of the parameter.

So far, this is just standard ReML. From line 167 to 168 k the derivative and hessian get adjusted to be relative to the exponential of $h$, which is necessary because the hyperparameters $h$ are parameterized in log-space to ensure positivity of the variance components. This adjustment accounts for the chain rule when transitioning from variance space ($v_i=exp(h_i)$) to log-space ($h_i$​), scaling the gradient and curvature appropriately to reflect the exponential relationship.

But then, from line 171, we combine the likelihood with the preferences, making the gradient and the curvature being the gradient and the curvature of hte free energy landscape. 

Based on `dFdh` (the free energy gradient) and `dFdhh` (free energy Fisher information matrix), we determine the values of `h` to try in the next iteration. We calculate `dh` at line 178 using spm_dx, and the estimated values get added to `h` at line 179. The `t` parameter controls how much regularization is applied: a low value means a lot of regularization, meaning that the prior is going to constrain the values of `dh` more. 

Following the estimate of the `h` values for the next iteration, we adjust the regularization parameter `t` for the next iteration. The idea is that if we see that change in free energy in this iteration is larger than in the previous, we have overshot and gone too far, so we should regularize more strongly on the next iteration to avoid overshooting again. In other words, if we are close to the peak, we want to progress more slowly, cause if we overshoot we might land further away from it than we already are.

The last step in the estimation loop is to determine whether to break it (line 195-202): if we don't make that much progress in free energy anymore, we consider the estimates to be good enough and we proceed with them. In addition, for covariance component whose estimated h weights are lower than the prior, we eliminate them, considering that they don't contribute anything, to avoid having these components messing up with the estimation of the other.

### Line 207-226: Calculating free energy
This section of the code calculates the free energy. We have provided a formulae in the previous notebook, but we use a slightly different though equivalent parametrization. We first compute the conditional precision of our estimate `Ph`, i.e. the inverse of the covariance matrix of our `h` estimates. From there onwards, it is quite straight forward: we calculate the complexity (line 216), the accuracy (line 220) and put the two together (and add some constant reflecting the degree of freedom and the normalization we applied at line 86-93) and that gives us the free energy.

Finally, from line 241-242, the estimated `h` and `C` (i.e. the residual covariance not explained by out model) are scaled to account for the normalization step applied at line 86-93 returning the provided estimates in the same original space as the inputed data 